In [5]:
!pip install yfinance
import pandas as pd
import numpy as np
import yfinance as yf

# 1. Định nghĩa danh sách các file cần bổ sung feature
bank_files = ["JPM_master_trees.csv", "WFC_master_trees.csv", "BAC_master_trees.csv"]

# 2. Tải dữ liệu XLF và chỉ số S&P 500 từ Yahoo Finance
print("=== Đang tải dữ liệu XLF và S&P 500 từ Yahoo Finance ===")
# Thêm group_by='ticker' và auto_adjust=False để cố định cấu trúc dữ liệu trả về
xlf_raw = yf.download("XLF", start="2019-11-01", end="2026-01-01")
spx_raw = yf.download("^GSPC", start="2019-11-01", end="2026-01-01")

# Ép phẳng cấu trúc cột (MultiIndex) nếu yfinance trả về dạng cột lồng nhau
if isinstance(xlf_raw.columns, pd.MultiIndex):
    xlf_raw.columns = xlf_raw.columns.get_level_values(0)
if isinstance(spx_raw.columns, pd.MultiIndex):
    spx_raw.columns = spx_raw.columns.get_level_values(0)

# Tìm chính xác tên cột chứa chữ 'Close' để tránh sai sót viết hoa/thường hay dấu cách
xlf_close_col = [c for c in xlf_raw.columns if 'Adj' in c or 'Close' in c][0]
spx_close_col = [c for c in spx_raw.columns if 'Adj' in c or 'Close' in c][0]

xlf_df = xlf_raw[[xlf_close_col]].rename(columns={xlf_close_col: 'XLF_Adj_Close'})
spx_df = spx_raw[[spx_close_col]].rename(columns={spx_close_col: 'SPX_Adj_Close'})

# Tạo một DataFrame chung cho thị trường vĩ mô và tính toán Log Return
market_data = xlf_df.join(spx_df, how='inner')
market_data['XLF_return'] = np.log(market_data['XLF_Adj_Close'] / market_data['XLF_Adj_Close'].shift(1))
market_data['SPX_return_macro'] = np.log(market_data['SPX_Adj_Close'] / market_data['SPX_Adj_Close'].shift(1))

# Tính các tính năng đặc trưng của Ngành (Sector-specific Features từ XLF)
market_data['XLF_volatility_5'] = market_data['XLF_return'].rolling(window=5).std()
market_data['XLF_momentum_5'] = market_data['XLF_Adj_Close'] / market_data['XLF_Adj_Close'].shift(5)

# Hàm tính toán chỉ báo RSI(14) thủ công
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))

# 3. Vòng lặp xử lý cho từng tập dữ liệu ngân hàng
for file_name in bank_files:
    print(f"\n--- Đang xử lý file: {file_name} ---")
    try:
        df = pd.read_csv(file_name)
    except FileNotFoundError:
        print(f"Không tìm thấy file {file_name}. Hãy chắc chắn bạn đã upload file lên Colab.")
        continue

    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)

    # Kiểm tra xem file của bạn dùng 'Adj_Close' hay 'Adj Close'
    bank_adj_col = 'Adj_Close' if 'Adj_Close' in df.columns else 'Adj Close'
    print(f"Sử dụng cột giá '{bank_adj_col}' của ngân hàng để tính chỉ báo kỹ thuật.")

    # --- PHẦN 1: TÍNH CÁC CHỈ BÁO KỸ THUẬT ---
    df['RSI_14'] = compute_rsi(df[bank_adj_col], period=14)

    # Tính MACD
    ema_12 = df[bank_adj_col].ewm(span=12, adjust=False).mean()
    ema_26 = df[bank_adj_col].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema_12 - ema_26
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Histogram'] = df['MACD'] - df['MACD_Signal']

    # --- PHẦN 2: TÍCH HỢP BIẾN NGÀNH XLF ---
    df = df.join(market_data[['XLF_return', 'XLF_volatility_5', 'XLF_momentum_5', 'SPX_return_macro']], how='left')

    # --- PHẦN 3: TÍNH TOÁN ROLLING BETA (30 ngày) ---
    # Tự động nhận diện cột return 1 ngày của ngân hàng (return_1d)
    bank_ret_col = 'return_1d' if 'return_1d' in df.columns else 'return'
    rolling_cov = df[bank_ret_col].rolling(window=30).cov(df['SPX_return_macro'])
    rolling_var = df['SPX_return_macro'].rolling(window=30).var()
    df['Rolling_Beta_30'] = rolling_cov / rolling_var

    # Loại bỏ biến đệm thị trường thừa
    df.drop(columns=['SPX_return_macro'], inplace=True, errors='ignore')

    # --- PHẦN 4: LÀM SẠCH VÀ XUẤT FILE ---
    df_clean = df.dropna()
    df_clean = df_clean.reset_index()

    output_name = file_name.replace(".csv", "_updated.csv")
    df_clean.to_csv(output_name, index=False)
    print(f"➔ Đã cập nhật xong! Lưu cấu trúc mới (Shape: {df_clean.shape}) vào file: {output_name}")

print("\n=== HOÀN THÀNH TOÀN BỘ QUY TRÌNH ===")

/tmp/ipykernel_1158/651993848.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  xlf_raw = yf.download("XLF", start="2019-11-01", end="2026-01-01")
[*********************100%***********************]  1 of 1 completed

=== Đang tải dữ liệu XLF và S&P 500 từ Yahoo Finance ===



/tmp/ipykernel_1158/651993848.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spx_raw = yf.download("^GSPC", start="2019-11-01", end="2026-01-01")
[*********************100%***********************]  1 of 1 completed



--- Đang xử lý file: JPM_master_trees.csv ---
Sử dụng cột giá 'Adj_Close' của ngân hàng để tính chỉ báo kỹ thuật.
➔ Đã cập nhật xong! Lưu cấu trúc mới (Shape: (1458, 54)) vào file: JPM_master_trees_updated.csv

--- Đang xử lý file: WFC_master_trees.csv ---
Sử dụng cột giá 'Adj_Close' của ngân hàng để tính chỉ báo kỹ thuật.
➔ Đã cập nhật xong! Lưu cấu trúc mới (Shape: (1458, 54)) vào file: WFC_master_trees_updated.csv

--- Đang xử lý file: BAC_master_trees.csv ---
Sử dụng cột giá 'Adj_Close' của ngân hàng để tính chỉ báo kỹ thuật.
➔ Đã cập nhật xong! Lưu cấu trúc mới (Shape: (1458, 54)) vào file: BAC_master_trees_updated.csv

=== HOÀN THÀNH TOÀN BỘ QUY TRÌNH ===
